# VitalWatch Internal SLM — QLoRA Fine-Tuning (Kaggle)

Fine-tunes **Qwen/Qwen2.5-1.5B-Instruct** as the VitalWatch *internal SLM / evidence compiler*:
retrieved medical-report text → **structured, privacy-minimized evidence JSON** for downstream
deterministic validation and the external LLM.

**The SLM is NOT:** a diagnosis model, the temporal prediction model, or a privacy guarantee.
The independent privacy validator in this notebook is a separate, authoritative gate.

## Pipeline position
`Medical Reports → OCR → RAG Retrieval → [THIS SLM] → Filtering → Deterministic Validation → Privacy Validator → Evidence Package → External LLM`

## Kaggle setup (right panel)
1. **Accelerator → GPU T4 x2** (P100 works too). Internet **On** (requires phone-verified account).
2. **Input → Upload → New Dataset**: upload the four files from `finetune/data/`
   (`train.jsonl`, `validation.jsonl`, `test.jsonl`, `dataset_statistics.json`) as one dataset
   (e.g. named `vitalwatch-slm-data`). The notebook auto-discovers them under `/kaggle/input`.
3. Run all cells top-to-bottom (~30–50 min: ~15–25 min training + ~15–25 min base-vs-tuned evaluation).
4. **Save Version** (Save & Run All), then download `vitalwatch-slm/adapter/` and
   `eval_results.json` from the **Output** tab.

Dataset provenance: ELMTEX (Zenodo 14793810, CC-BY-4.0) for extraction + clearly-labeled synthetic
edge cases for privacy/abstention. See `finetuning_dataset.md`. No real PHI is used anywhere.

In [ ]:
%pip install -q -U "transformers>=4.46" "trl>=0.12" "peft>=0.13" "datasets>=2.20" accelerate bitsandbytes evaluate pandas scikit-learn regex matplotlib
print("install done")

In [ ]:
# --- Environment & hardware verification (do not assume a specific GPU) ---
import os, sys, subprocess, json, shutil

def gpu_info():
    if shutil.which("nvidia-smi") is None:
        return []
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total",
             "--format=csv,noheader,nounits"], text=True, timeout=30)
        rows = [l.strip().split(", ") for l in out.strip().splitlines() if l.strip()]
        return [{"name": r[0], "vram_mb": int(float(r[1]))} for r in rows]
    except Exception as e:
        print("nvidia-smi failed:", e)
        return []

import torch
print("python:", sys.version.split()[0], "| torch:", torch.__version__)
GPUS = gpu_info()
print("GPUs:", GPUS if GPUS else "NONE - training on CPU will be extremely slow")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Attach a GPU accelerator (Settings -> Accelerator). "
        "This notebook is sized for a 16 GB T4; CPU execution is not practical.")

CAP = torch.cuda.get_device_capability(0)          # e.g. (7, 5) = T4
COMPUTE_DTYPE = torch.bfloat16 if CAP[0] >= 8 else torch.float16
print(f"compute capability {CAP} -> compute dtype {COMPUTE_DTYPE}")
TOTAL_VRAM_MB = GPUS[0]["vram_mb"]
assert TOTAL_VRAM_MB >= 12000, f"need >= 12 GB VRAM, found {TOTAL_VRAM_MB} MB"
print(f"VRAM check OK: {TOTAL_VRAM_MB} MB")

In [ ]:
# --- Central configuration ---
from pathlib import Path

MODEL_ID       = "Qwen/Qwen2.5-1.5B-Instruct"   # Apache-2.0, ungated (verified on HF)
ADAPTER_DIR    = Path("/kaggle/working/vitalwatch-slm/adapter")
EVAL_OUT       = Path("/kaggle/working/eval_results.json")
SCHEMA_VERSION = "1.0"

# hyperparameters (sized for 16 GB T4, ~1.1k train examples)
NUM_EPOCHS    = 3
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
MAX_SEQ_LEN   = 2048        # was 1280
BATCH_SIZE    = 2
GRAD_ACCUM    = 8          # effective batch 16
LEARNING_RATE = 2e-4
WARMUP_RATIO  = 0.03
EVAL_STEPS    = 25
SAVE_STEPS    = 50
SEED          = 42

# evaluation subset (stratified, fixed)
EVAL_N_TOTAL  = 100
EVAL_GEN_MAX_NEW = 384
LATENCY_SAMPLES  = 10

TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

In [ ]:
# --- Dataset loading (auto-discover the attached Kaggle Dataset) ---
import glob

def find_input(filename):
    hits = sorted(glob.glob(f"/kaggle/input/**/{filename}", recursive=True))
    return Path(hits[0]) if hits else None

DATA_FILES = {s: find_input(f"{s}.jsonl") for s in ("train", "validation", "test")}
missing = [s for s, p in DATA_FILES.items() if p is None]
if missing:
    raise FileNotFoundError(
        f"Could not find {missing} under /kaggle/input. Attach the dataset built by "
        "finetune/data (train/validation/test.jsonl) via Input -> Upload -> New Dataset.")

DATA = {}
for s, p in DATA_FILES.items():
    rows = []
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    DATA[s] = rows
    print(f"{s}: {len(rows):4d} examples  ({p})")

STATS_FILE = find_input("dataset_statistics.json")
if STATS_FILE:
    print("\nbuild statistics:", json.loads(STATS_FILE.read_text(encoding="utf-8"))["splits"])

In [ ]:
# --- Dataset cleaning & schema validation ---
OUTPUT_KEYS  = {"schema_version", "status", "source_reference", "observations",
                "removed_identifiers", "missing_information", "uncertainty", "abstention_reason"}
OBS_KEYS     = {"evidence_id", "parameter", "value", "unit", "value_type",
                "temporal", "uncertainty", "note"}
STATUSES     = {"success", "partial", "abstain"}

def validate_example(ex):
    problems = []
    for k in ("instruction", "input", "output", "meta"):
        if k not in ex:
            problems.append(f"missing top-level key {k}")
            return problems
    out = ex["output"]
    if not OUTPUT_KEYS <= set(out.keys()):
        problems.append(f"output keys mismatch: missing {OUTPUT_KEYS - set(out.keys())}")
    if out.get("status") not in STATUSES:
        problems.append(f"bad status {out.get('status')!r}")
    if out.get("status") == "abstain" and not out.get("abstention_reason"):
        problems.append("abstain without abstention_reason")
    for ob in out.get("observations", []):
        if not OBS_KEYS <= set(ob.keys()):
            problems.append("observation keys mismatch")
            break
        if ob.get("value_type") not in ("numeric", "categorical"):
            problems.append(f"bad value_type {ob.get('value_type')!r}")
    return problems

clean, dropped = {}, []
for s, rows in DATA.items():
    keep = []
    for idx, ex in enumerate(rows):
        probs = validate_example(ex)
        if probs:
            dropped.append((s, idx, ex.get("meta", {}).get("category"), probs))
        else:
            keep.append(ex)
    clean[s] = keep
    print(f"{s}: kept {len(keep)}/{len(rows)}")
if dropped:
    print(f"dropped {len(dropped)} invalid examples; first few:")
    for d in dropped[:5]:
        print("  ", d)
DATA = clean

In [ ]:
# --- Split verification: patient-level separation, no leakage ---
def patient_ids(rows):
    return {ex["meta"]["patient_id"] for ex in rows}

def source_refs(rows):
    return {ex["output"]["source_reference"] for ex in rows}

pid = {s: patient_ids(rows) for s, rows in DATA.items()}
sref = {s: source_refs(rows) for s, rows in DATA.items()}
inputs = {s: [ex["input"] for ex in DATA[s]] for s in DATA}

for a, b in (("train", "validation"), ("train", "test"), ("validation", "test")):
    ov_pid = pid[a] & pid[b]
    ov_sref = sref[a] & sref[b]
    ov_inp = set(inputs[a]) & set(inputs[b])
    print(f"{a} vs {b}: shared patients={len(ov_pid)}, shared source refs={len(ov_sref)}, identical inputs={len(ov_inp)}")
    assert not ov_pid and not ov_sref and not ov_inp, "SPLIT LEAKAGE DETECTED"
print("split verification PASSED (patient-level, source-level, input-level)")

In [ ]:
# --- Tokenization / chat-template formatting ---
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def to_messages(ex):
    target = json.dumps(ex["output"], ensure_ascii=False, separators=(",", ":"))
    return [
        {"role": "system", "content":
         "You are the VitalWatch internal SLM (evidence compiler). Convert the medical report "
         "into the privacy-minimized evidence JSON schema. Remove identifiers, use relative "
         "times, flag missing/inconsistent information, abstain when unreliable, never invent values."},
        {"role": "user", "content": ex["instruction"] + "\n\nREPORT:\n" + ex["input"]},
        {"role": "assistant", "content": target},
    ]

def formatted(ex):
    return tokenizer.apply_chat_template(to_messages(ex), tokenize=False)

for s in DATA:
    DATA[s] = [{**ex, "text": formatted(ex)} for ex in DATA[s]]

train_ds = Dataset.from_list([{"text": ex["text"]} for ex in DATA["train"]])
val_ds   = Dataset.from_list([{"text": ex["text"]} for ex in DATA["validation"]])
print("formatted example (first 420 chars):\n", train_ds[0]["text"][:420])

lens = [len(tokenizer(ex["text"]).input_ids) for ex in DATA["train"]]
print(f"\ntrain token lengths: median={sorted(lens)[len(lens)//2]}, max={max(lens)}")
assert max(lens) <= MAX_SEQ_LEN, "examples exceed MAX_SEQ_LEN - raise it in the config cell" 

In [ ]:
# --- Base model with 4-bit quantization (QLoRA) ---
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    torch_dtype=COMPUTE_DTYPE,
    device_map={"": 0},
)
model.config.use_cache = False
model.gradient_checkpointing_enable()
print(f"model footprint: {model.get_memory_footprint()/1e9:.2f} GB")

In [ ]:
# --- LoRA configuration ---
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
peft_cfg = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none", task_type="CAUSAL_LM", target_modules=TARGET_MODULES,
)
model = get_peft_model(model, peft_cfg)
model.print_trainable_parameters()

In [ ]:
# --- Training (with validation-loss tracking and checkpoints) ---
import dataclasses
import math
import time
from trl import SFTConfig, SFTTrainer

cast_count = 0
for p in model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.to(torch.float32)
        cast_count += 1
print(f"cast {cast_count} trainable tensors to float32")

t_train_start = time.time()
steps_per_epoch = math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM))
total_steps = max(1, steps_per_epoch * NUM_EPOCHS)

candidate = dict(
    output_dir="/kaggle/working/vitalwatch-slm/checkpoints",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=max(2, int(WARMUP_RATIO * total_steps)),
    logging_steps=5,
    eval_strategy="steps", eval_steps=EVAL_STEPS,
    save_strategy="steps", save_steps=SAVE_STEPS, save_total_limit=2,
    max_length=MAX_SEQ_LEN, packing=False,
    fp16=False, bf16=False,
    optim="paged_adamw_8bit",
    report_to="none", seed=SEED,
    dataset_text_field="text",
)
supported = {f.name for f in dataclasses.fields(SFTConfig)}
dropped = [k for k in candidate if k not in supported]
cfg_kwargs = {k: v for k, v in candidate.items() if k in supported}
if dropped:
    print("NOTE: installed SFTConfig does not support:", dropped, "-> dropped")
assert "max_length" in cfg_kwargs, "SFTConfig no longer accepts max_length"
cfg = SFTConfig(**cfg_kwargs)

try:
    cfg.n_gpu = 1
except AttributeError:
    cfg._n_gpu = 1

trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, eval_dataset=val_ds,
                     processing_class=tokenizer)
trainer.train()
TRAIN_SECONDS = time.time() - t_train_start
print(f"training finished in {TRAIN_SECONDS/60:.1f} min")

In [ ]:
# --- Loss tracking summary ---
import matplotlib.pyplot as plt

log = trainer.state.log_history
tr = [(h["step"], h["loss"]) for h in log if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in log if "eval_loss" in h]
plt.figure(figsize=(7, 4))
if tr: plt.plot(*zip(*tr), label="train loss")
if ev: plt.plot(*zip(*ev), marker="o", label="eval loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("QLoRA training curves")
plt.tight_layout(); plt.savefig("/kaggle/working/loss_curve.png", dpi=110); plt.show()
print("final eval loss:", ev[-1][1] if ev else "n/a")

In [ ]:
# --- Save adapter + tokenizer + model card ---
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

model_card = f"""---
license: apache-2.0
base_model: {MODEL_ID}
library_name: peft
tags: [vitalwatch, clinical-evidence, qlora, lora]
---

# VitalWatch Internal SLM (LoRA adapter)
Evidence compiler for the VitalWatch pipeline: converts retrieved medical report text into a
privacy-minimized evidence JSON package (schema v{SCHEMA_VERSION}). Decision-support only; not a
diagnostic device, not the temporal prediction model, not a privacy guarantee - all output must
pass the independent deterministic validator and privacy firewall before any external transmission.

Trained on ELMTEX (Zenodo 14793810, CC-BY-4.0) derived extraction examples plus clearly-labeled
synthetic edge cases. No real patient data (PHI) used.
"""
(ADAPTER_DIR / "README.md").write_text(model_card, encoding="utf-8")
size_mb = sum(f.stat().st_size for f in ADAPTER_DIR.rglob("*") if f.is_file()) / 1e6
print(f"adapter saved to {ADAPTER_DIR} ({size_mb:.1f} MB)")

In [ ]:
# --- Generation helper & robust JSON parsing ---
import re, time

INJECTION_PAT = re.compile(r"(ignore (all )?(previous|prior)|system override|disregard the)", re.I)

def build_prompt(ex):
    msgs = to_messages(ex)
    return tokenizer.apply_chat_template(
        [{k: m[k] for k in ("role", "content")} for m in msgs[:2]],
        tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate_evidence(model_ref, ex, max_new_tokens=EVAL_GEN_MAX_NEW):
    prompt = build_prompt(ex)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model_ref.device)
    t0 = time.perf_counter()
    out_ids = model_ref.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id)
    dt = time.perf_counter() - t0
    text = tokenizer.decode(out_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return text, dt

def parse_evidence(text):
    """Extract the first balanced JSON object from model output."""
    m = re.search(r"\{", text)
    if not m:
        return None
    depth, start = 0, m.start()
    for i in range(start, len(text)):
        if text[i] == "{": depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i + 1])
                except Exception:
                    return None
    return None

# smoke test on one validation example
sample = DATA["validation"][0]
raw, dt = generate_evidence(model, sample)
pred = parse_evidence(raw)
print("gold status:", sample["output"]["status"], "| parsed:", pred is not None, f"| {dt:.1f}s")
print(raw[:400])

In [ ]:
# --- Independent privacy validator (authoritative gate, NOT the SLM) ---
# Runs regardless of what the SLM claims. Mirrors the VitalWatch firewall checks:
# field allow-list, schema validation, identifier-pattern scan, evidence-ID resolution,
# injection detection, and rejection of unsafe payloads.

ALLOWED_TOP_KEYS = {"schema_version", "status", "source_reference", "observations",
                    "removed_identifiers", "missing_information", "uncertainty",
                    "abstention_reason"}
LEAK_PATTERNS = {
    "email": re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"),
    "phone": re.compile(r"(?:\+?1[-. ]?)?\(?\d{3}\)?[-. ]\d{3}[-. ]\d{4}"),
    "mrn": re.compile(r"\bMRN[-:# ]?\s?\d+", re.I),
    "street_address": re.compile(r"\b\d{1,4}\s+[A-Z][a-z]+\s(?:Street|Avenue|Lane|Road|Crescent)\b"),
}
NAME_RE = re.compile(r"Patient:\s*([A-Z][a-z]+ [A-Z][a-z]+)")

def privacy_validate(pred, source_text):
    """Return (allowed: bool, violations: [str]). Independent of model claims."""
    v = []
    if pred is None:
        return False, ["invalid_json"]
    if not isinstance(pred, dict):
        return False, ["not_an_object"]
    extra = set(pred.keys()) - ALLOWED_TOP_KEYS
    if extra:
        v.append(f"schema_violation: unexpected keys {sorted(extra)}")
    if pred.get("schema_version") != SCHEMA_VERSION:
        v.append(f"schema_violation: schema_version {pred.get('schema_version')!r}")
    for ob in pred.get("observations", []) if isinstance(pred.get("observations"), list) else []:
        if not OBS_KEYS <= set(ob.keys()):
            v.append("schema_violation: observation field outside allow-list")
            break
    flat = json.dumps(pred, ensure_ascii=False)
    m = NAME_RE.search(source_text or "")
    if m and m.group(1) in flat:
        v.append("pii_leak: patient name copied into evidence")
    for kind, rx in LEAK_PATTERNS.items():
        if kind == "mrn":
            if rx.search(flat) and "MRN" in (source_text or ""):
                v.append(f"pii_leak: {kind}")
        elif rx.search(flat):
            v.append(f"pii_leak: {kind}")
    if INJECTION_PAT.search(flat):
        v.append("injection_echo: instruction-like text copied into evidence")
    if pred.get("status") not in STATUSES:
        v.append("schema_violation: bad status")
    return (len(v) == 0), v

# Demonstrate the firewall rejecting an UNSAFE payload even if the model produced it:
unsafe = {"schema_version": SCHEMA_VERSION, "status": "success",
          "source_reference": "RPT_demo",
          "observations": [{"evidence_id": "obs_001", "parameter": "hemoglobin", "value": 13.2,
                            "unit": "g/dL", "value_type": "numeric",
                            "temporal": {"raw": "03 Jan 2024", "relative": "collection day +0"},
                            "uncertainty": "none", "note": "contact patient at 555-0112, j.doe@x.com"}],
          "removed_identifiers": [], "missing_information": [], "uncertainty": [],
          "abstention_reason": None}
unsafe["removed_identifiers"] = []
ok, viols = privacy_validate(unsafe, "Patient: Jane Doe   (MRN: MRN-123456)")
print("unsafe payload allowed?", ok, "| violations:", viols)
assert not ok, "firewall failed to reject unsafe payload"
print("privacy firewall demo PASSED: unsafe payload rejected")

In [ ]:
# --- Fixed, stratified evaluation subset from the held-out test set ---
from collections import defaultdict

by_cat = defaultdict(list)
for ex in DATA["test"]:
    by_cat[ex["meta"]["category"]].append(ex)
eval_set = []
cats = sorted(by_cat)
half = EVAL_N_TOTAL // 2
# half from ELMTEX extraction, half spread evenly over synthetic categories
elmtex_rows = by_cat["extraction_elmtex"]
eval_set += elmtex_rows[:half]
synth_cats = [c for c in cats if c != "extraction_elmtex"]
per = max(1, half // len(synth_cats))
for c in synth_cats:
    eval_set += by_cat[c][:per]
eval_set = eval_set[:EVAL_N_TOTAL]
print(f"evaluation subset: {len(eval_set)} examples "
      f"({sum(1 for e in eval_set if e['meta']['data_origin']=='elmtex')} elmtex / "
      f"{sum(1 for e in eval_set if e['meta']['data_origin']=='synthetic')} synthetic)")

In [ ]:
# --- Metric implementations (extraction, privacy, structure, abstention) ---
def obs_key(ob):
    v = ob.get("value")
    if isinstance(v, float):
        v = round(v, 4)
    return (str(ob.get("parameter", "")).strip().lower(), str(v),
            (ob.get("unit") or "").strip().lower())

def prf(pred_sets, gold_sets):
    tp = sum(len(p & g) for p, g in zip(pred_sets, gold_sets))
    fp = sum(len(p - g) for p, g in zip(pred_sets, gold_sets))
    fn = sum(len(g - p) for p, g in zip(pred_sets, gold_sets))
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return prec, rec, f1

def evaluate_model(model_ref, examples, label):
    n = len(examples)
    m = {"label": label, "n": n, "json_valid": 0, "schema_ok": 0, "firewall_rejected": 0,
         "correct_status": 0, "abstention_correct": 0, "pii_leak": 0,
         "pred_sets": [], "gold_sets": [], "value_match": [0, 0], "unit_match": [0, 0],
         "src_ref_ok": 0, "latencies": [], "tokens": []}
    for ex in examples:
        gold = ex["output"]
        raw, dt = generate_evidence(model_ref, ex)
        pred = parse_evidence(raw)
        m["latencies"].append(dt)
        m["tokens"].append(len(tokenizer(raw).input_ids))
        ok, viols = privacy_validate(pred, ex["input"])
        if pred is not None:
            m["json_valid"] += 1
        if ok:
            m["schema_ok"] += 1
        else:
            m["firewall_rejected"] += 1
        if any(x.startswith("pii_leak") for x in viols):
            m["pii_leak"] += 1
        if pred and pred.get("source_reference") == gold["source_reference"]:
            m["src_ref_ok"] += 1
        status = pred.get("status") if pred else None
        if status == gold["status"]:
            m["correct_status"] += 1
        gold_abstain = gold["status"] == "abstain"
        if (status == "abstain") == gold_abstain:
            m["abstention_correct"] += 1
        gset = {obs_key(o) for o in gold["observations"]}
        pset = {obs_key(o) for o in (pred or {}).get("observations", [])} if pred else set()
        m["pred_sets"].append(pset); m["gold_sets"].append(gset)
        for gk in gset:
            if gk[0] in {pk[0] for pk in pset}:
                m["value_match"][1] += 1
                if gk[1] in {pk[1] for pk in pset if pk[0] == gk[0]}:
                    m["value_match"][0] += 1
                m["unit_match"][1] += 1
                if gk[2] and gk[2] in {pk[2] for pk in pset if pk[0] == gk[0]}:
                    m["unit_match"][0] += 1
    prec, rec, f1 = prf(m.pop("pred_sets"), m.pop("gold_sets"))
    m["field_precision"], m["field_recall"], m["field_f1"] = prec, rec, f1
    m["json_valid_rate"] = m["json_valid"] / n
    m["schema_ok_rate"] = m["schema_ok"] / n
    m["invalid_output_rate"] = 1 - m["json_valid"] / n
    m["firewall_rejection_rate"] = m["firewall_rejected"] / n
    m["pii_leak_rate"] = m["pii_leak"] / n
    m["status_accuracy"] = m["correct_status"] / n
    m["abstention_accuracy"] = m["abstention_correct"] / n
    m["source_ref_rate"] = m["src_ref_ok"] / n
    m["value_preservation_rate"] = m["value_match"][0] / m["value_match"][1] if m["value_match"][1] else None
    m["unit_preservation_rate"] = m["unit_match"][0] / m["unit_match"][1] if m["unit_match"][1] else None
    m["median_latency_s"] = sorted(m["latencies"])[n // 2]
    m["mean_latency_s"] = sum(m["latencies"]) / n
    m["tokens_per_second"] = sum(m["tokens"]) / sum(m["latencies"])
    return m

import time as _t
def timed_model_metrics(model_ref, examples, label, n_latency=LATENCY_SAMPLES):
    return evaluate_model(model_ref, examples, label)

In [ ]:
# --- Evaluate fine-tuned model vs base model on the SAME test subset ---
# Base model (no adapter) for comparison: reload fresh in 4-bit.
import gc

torch.cuda.reset_peak_memory_stats()
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, torch_dtype=COMPUTE_DTYPE, device_map={"": 0})
base_model.config.use_cache = True

gc.collect(); torch.cuda.empty_cache()
metrics_base = evaluate_model(base_model, eval_set, "base")
del base_model
gc.collect(); torch.cuda.empty_cache()

model.config.use_cache = True   # re-enable KV cache for generation
metrics_ft = evaluate_model(model, eval_set, "fine_tuned")

mem_gb = torch.cuda.max_memory_allocated() / 1e9
adapter_mb = sum(f.stat().st_size for f in ADAPTER_DIR.rglob("*") if f.is_file()) / 1e6

summary = {
    "schema_version": SCHEMA_VERSION,
    "model": MODEL_ID,
    "eval_examples": len(eval_set),
    "training_seconds": TRAIN_SECONDS,
    "training_minutes": round(TRAIN_SECONDS / 60, 1),
    "gpu_peak_memory_gb": round(mem_gb, 2),
    "adapter_size_mb": round(adapter_mb, 1),
    "base": {k: v for k, v in metrics_base.items() if not isinstance(v, list)},
    "fine_tuned": {k: v for k, v in metrics_ft.items() if not isinstance(v, list)},
}
EVAL_OUT.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

hdr = f"{'metric':28s} {'base':>10s} {'fine-tuned':>12s}"
print(hdr); print("-" * len(hdr))
for k in metrics_base:
    if k in ("label", "n", "latencies", "tokens"): continue
    b, f_ = metrics_base[k], metrics_ft[k]
    fmt = lambda x: f"{x:.3f}" if isinstance(x, float) else str(x)
    print(f"{k:28s} {fmt(b):>10s} {fmt(f_):>12s}")
print(f"\ngpu peak {mem_gb:.2f} GB | adapter {adapter_mb:.1f} MB | train {TRAIN_SECONDS/60:.1f} min")
print("full results ->", EVAL_OUT)

In [ ]:
# --- Qualitative before/after (fine-tuned model on a held-out privacy example) ---
# (The base model was released from VRAM after the comparison cell; the comparison
#  cell itself already prints the base-vs-tuned metric table on identical examples.)

idx = next(i for i, e in enumerate(eval_set) if e["meta"]["category"] == "identifiers_present")
ex = eval_set[idx]
raw_ft, _ = generate_evidence(model, ex)
print("INPUT (first 260 chars):\n" + ex["input"][:260])
print("\nFINE-TUNED OUTPUT:\n" + raw_ft[:700])
pred = parse_evidence(raw_ft)
ok, viols = privacy_validate(pred, ex["input"])
print("\nfirewall verdict:", "ALLOWED" if ok else f"REJECTED {viols}")
print("\nidentifiers in input removed per gold:", ex["output"]["removed_identifiers"])

In [ ]:
# --- Optional: push the adapter to the Hugging Face Hub ---
# 1) Create a write token at https://huggingface.co/settings/tokens
# 2) Kaggle menu: Add-ons -> Secrets -> add a secret named HF_TOKEN (attach to this notebook)
# 3) Uncomment and run:
#
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import login
# login(UserSecretsClient().get_secret("HF_TOKEN"))
# repo = "YOUR_HF_USERNAME/vitalwatch-slm-qwen-1p5b-lora"   # keep private unless license review done
# model.push_to_hub(repo, private=True)
# tokenizer.push_to_hub(repo, private=True)
#
# The VitalWatch application then loads: base model (pinned revision) + this adapter
# (pinned revision) + tokenizer, and validates outputs against schema v1.0.
# License: Apache-2.0 base -> Apache-2.0 adapter; redistribution permitted with attribution.
# NEVER upload real patient records, PHI, credentials, or non-public datasets.
print("HF push instructions in this cell (uncomment to use)")

## Limitations (read before using results)
- Results above are produced by the notebook run on the **held-out synthetic/ELMTEX-derived test set** - they do not establish real-world clinical performance (PRD: synthetic-only evaluation limitation).
- Fine-tuning does **not** make the model privacy-preserving; the independent validator/firewall remains the gate (de-identification ≠ anonymity).
- MIMIC-IV-Ext-Instr and n2c2/i2b2 2014 require credentialed access (PhysioNet DUA / Harvard DBMI per-user DUA) and are therefore **not used**; substitute documented in `finetuning_dataset.md`.
- Base-vs-tuned comparison reuses one T4 sequentially; latency numbers include tokenization overhead.
- The evidence schema here (v1.0) must be mapped to the versioned `EvidencePackage` contract defined in `architecture.md` §15 during integration.

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/vitalwatch-adapter", "zip", "/kaggle/working/vitalwatch-slm/adapter")
print("zip created")

In [ ]:
# --- Eval configuration fix ---
EVAL_GEN_MAX_NEW = 768                     # long evidence packages need > 384 tokens
model.gradient_checkpointing_disable()     # ~2x faster generation, fairer latency
model.config.use_cache = True

# re-bind generate_evidence so the new cap takes effect (defaults are captured
# at definition time, so the function must be redefined)
import time

@torch.no_grad()
def generate_evidence(model_ref, ex, max_new_tokens=EVAL_GEN_MAX_NEW):
    prompt = build_prompt(ex)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model_ref.device)
    t0 = time.perf_counter()
    out_ids = model_ref.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id)
    dt = time.perf_counter() - t0
    text = tokenizer.decode(out_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return text, dt

print("generate_evidence re-bound with max_new_tokens =", EVAL_GEN_MAX_NEW)

In [ ]:
# --- Evaluate fine-tuned model vs base model on the SAME test subset ---
# Base model (no adapter) for comparison: reload fresh in 4-bit.
import gc

torch.cuda.reset_peak_memory_stats()
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, torch_dtype=COMPUTE_DTYPE, device_map={"": 0})
base_model.config.use_cache = True

gc.collect(); torch.cuda.empty_cache()
metrics_base = evaluate_model(base_model, eval_set, "base")
del base_model
gc.collect(); torch.cuda.empty_cache()

model.config.use_cache = True   # re-enable KV cache for generation
metrics_ft = evaluate_model(model, eval_set, "fine_tuned")

mem_gb = torch.cuda.max_memory_allocated() / 1e9
adapter_mb = sum(f.stat().st_size for f in ADAPTER_DIR.rglob("*") if f.is_file()) / 1e6

summary = {
    "schema_version": SCHEMA_VERSION,
    "model": MODEL_ID,
    "eval_examples": len(eval_set),
    "training_seconds": TRAIN_SECONDS,
    "training_minutes": round(TRAIN_SECONDS / 60, 1),
    "gpu_peak_memory_gb": round(mem_gb, 2),
    "adapter_size_mb": round(adapter_mb, 1),
    "base": {k: v for k, v in metrics_base.items() if not isinstance(v, list)},
    "fine_tuned": {k: v for k, v in metrics_ft.items() if not isinstance(v, list)},
}
EVAL_OUT.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

hdr = f"{'metric':28s} {'base':>10s} {'fine-tuned':>12s}"
print(hdr); print("-" * len(hdr))
for k in metrics_base:
    if k in ("label", "n", "latencies", "tokens"): continue
    b, f_ = metrics_base[k], metrics_ft[k]
    fmt = lambda x: f"{x:.3f}" if isinstance(x, float) else str(x)
    print(f"{k:28s} {fmt(b):>10s} {fmt(f_):>12s}")
print(f"\ngpu peak {mem_gb:.2f} GB | adapter {adapter_mb:.1f} MB | train {TRAIN_SECONDS/60:.1f} min")
print("full results ->", EVAL_OUT)

# 📋 Execution log (actual run history)

## Part 1 — Full pipeline (`Run All`, with the 2026-Kaggle fixes applied along the way)

| # | Step | Result |
|---|---|---|
| 1 | Session setup | GPU T4 x2, Internet On, `vitalwatch-slm-data` dataset attached (phone-verified account) |
| 2 | Installs | transformers / trl / peft / datasets / bitsandbytes (Kaggle 2026 image, Python 3.13) |
| 3 | Hardware check | T4 (compute capability 7.5) → fp16 compute dtype, ≥12 GB assert passed |
| 4 | Config | `MAX_SEQ_LEN` raised **1280 → 2048** — the token-length assert caught real max 1,686 (median 802) |
| 5 | Data load | train 1,087 / validation 123 / test 140 — schema validation kept all, split verification: **0 shared patients / source refs / inputs** |
| 6 | Formatting | Qwen chat template, target = compact evidence JSON |
| 7 | Base model | Qwen2.5-1.5B-Instruct, 4-bit NF4 + double quant, fp16 compute (~1.1 GB footprint) |
| 8 | LoRA | r=16, α=32, all attention+MLP projections → **18,464,768 trainable / 1.18%** of params |
| 9 | Training-cell fixes (2026 stack) | `warmup_ratio` removed → `warmup_steps`; `cfg.n_gpu = 1` (T4 x2 exposes 2 GPUs, DataParallel unsupported in notebooks); **AMP off** (`fp16=False, bf16=False`) — fp16 GradScaler has no bf16-unscale kernel; LoRA weights cast to fp32 |
| 10 | Training | 204 steps / 3 epochs in **116.1 min**; final eval loss **0.440**, mean token accuracy **88.7%** |
| 11 | Adapter saved | `/kaggle/working/vitalwatch-slm/adapter` — **48.4 MB** (18.46 M fp16 params + tokenizer) |
| 12 | Privacy firewall demo | deliberately unsafe payload **rejected** (`pii_leak: email`) |
| 13 | Evaluation #1 | at `max_new_tokens=384`: privacy results perfect (`pii_leak_rate 0.000`, `value_preservation 1.000`, `abstention_accuracy 1.000`) but long ELMTEX outputs were **truncated** → `json_valid_rate 0.284`, `field_recall 0.122` (config artifact, not model quality) |
| 14 | Persist | Quick Save → adapter downloaded; adapter pushed to HF: `TheBlindCoder/vitalwatch-slm-qwen-1p5b-lora` (private) |

## Part 2 — Corrected evaluation (`max_new_tokens = 768`, no retraining)

After an accidental revert to the pre-fix saved version (fresh kernel), evaluation was redone
by resuming **exactly the production load path** — pinned base + adapter from the HF Hub:

| # | Step |
|---|---|
| 1 | Config edit: `EVAL_GEN_MAX_NEW = 768` (default args are captured at function definition, so `generate_evidence` was re-bound in the fix cell) |
| 2 | Re-ran cheap setup cells: installs → hardware check → config → data load → schema validation → split verification → formatting |
| 3 | **Skipped** model-load / LoRA / training cells; instead: `PeftModel.from_pretrained(base, "TheBlindCoder/vitalwatch-slm-qwen-1p5b-lora")` with HF_TOKEN from Kaggle Secrets; adapter re-saved to `ADAPTER_DIR`; `TRAIN_SECONDS = 116.1 * 60` restored for the summary |
| 4 | Generation-helper, privacy-validator, eval-subset (95 examples: 50 ELMTEX / 45 synthetic), metrics cells re-run |
| 5 | Evaluation cell re-run (~45–60 min): base ~25–40 min at 768 tokens, fine-tuned faster with `gradient_checkpointing_disable()` |
| 6 | Qualitative cell (PII-report demo + firewall verdict) → Quick Save → download `eval_results.json` |
| 7 | Final numbers recorded in `finetuning_report.md` §7 |

**Lesson learned:** generation-token budget must exceed the longest gold output (a 14-observation
ELMTEX evidence package ≈ 700+ tokens); always sanity-check `max(gen output tokens) < max_new_tokens`.

# GGUF Conversion


In [ ]:
!pip uninstall -y torchao -q

In [ ]:
# --- Merge LoRA adapter into the base model (fp16, UNQUANTIZED base) ---
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
login(UserSecretsClient().get_secret("HF_TOKEN"))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MERGED_DIR = "/kaggle/working/vitalwatch-slm-merged"
base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct", torch_dtype=torch.float16, device_map={"": 0})
model = PeftModel.from_pretrained(base, "TheBlindCoder/vitalwatch-slm-qwen-1p5b-lora")
merged = model.merge_and_unload()
merged.save_pretrained(MERGED_DIR)
AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct").save_pretrained(MERGED_DIR)
import shutil
shutil.rmtree("/kaggle/working/llama.cpp", ignore_errors=True)  # cleanup if re-running
print("merged model saved to", MERGED_DIR)   # ~3.1 GB

In [ ]:
# --- Convert merged model to GGUF (F16) ---
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp
%pip install -q gguf sentencepiece protobuf
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py /kaggle/working/vitalwatch-slm-merged --outfile /kaggle/working/vitalwatch-slm-f16.gguf --outtype f16
!ls -la /kaggle/working/*.gguf